# Notebook 02: LLM Prompt Selection

Sensitivity analysis to determine which prompt formulation yields the
best classification performance for the main LLM evaluation in
Notebook 03.

## Methodology

**Question.** How should the 76 catalogue classes be described to the
LLM in the system prompt? Five formulations are compared empirically on
the same 64 labelled contract sections.

**Variants tested.**

| ID | Class definition |
|---|---|
| A | `section_topic` only |
| B | `paragraph_topic` + `section_topic` |
| C | `example` only (analog to the encoder reference text) |
| D | `section_topic` + `example` |
| E | `paragraph_topic` + `section_topic` + `example` (everything) |

**Lightweight evaluation.** Each variant is run as a single pass over
all 64 sections. No reasoning is requested, no stability runs are
performed, no bootstrap CIs are computed. The goal is to identify the
variant that maximises macro F1 on the evaluation set, not to fully
characterise its performance.

**A-priori expectation.** Variant B (`paragraph_topic + section_topic`)
was theoretically preferred because the section_topic field provides a
human-readable definition of the class, while the example field risks
surface-level string matching. If variant B is also empirically best
or close to best, the theoretical and empirical arguments converge.

**Methodological caveat.** This procedure constitutes in-sample model
selection on N=64 samples, since no separate validation set is available
given the limited corpus size. The selected variant is therefore
optimised against the same data used for the main evaluation in
Notebook 03. This limitation is discussed in Chapter 9.3 of the thesis.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "llm_prompt_selection"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LLM_MODEL = "gpt-5.5-2026-04-23"
SEED = 42
DELAY_BETWEEN_CALLS = 1.0

client = OpenAI()
print(f"Model: {LLM_MODEL}")

## Step 1: Load data

In [ ]:
sections = pd.read_excel(DATA_DIR / "sections_labeled_manually.xlsx")
sections = sections.drop(columns=["Unnamed: 0"], errors="ignore")

catalogue_raw = pd.read_excel(DATA_DIR / "catalogue_clean_with_aspects.xlsx")
catalogue = catalogue_raw[["paragraph_topic", "section_topic", "example"]].copy()
catalogue["catalog_id"] = catalogue.index + 1

print(f"Sections: {len(sections)}, Catalogue: {len(catalogue)}")

## Step 2: Define prompt variants

Each variant is a function that renders one catalogue row into a string
representation. The five renderings differ only in which catalogue
fields are included.

In [ ]:
def render_A_section_only(row: pd.Series) -> str:
    return f"[ID {row['catalog_id']}] {row['section_topic']}"


def render_B_paragraph_section(row: pd.Series) -> str:
    return (
        f"[ID {row['catalog_id']}] {row['paragraph_topic']}\n"
        f"        {row['section_topic']}"
    )


def render_C_example_only(row: pd.Series) -> str:
    return f"[ID {row['catalog_id']}] {row['example']}"


def render_D_section_example(row: pd.Series) -> str:
    return (
        f"[ID {row['catalog_id']}] {row['section_topic']}\n"
        f"        Example: {row['example']}"
    )


def render_E_all(row: pd.Series) -> str:
    return (
        f"[ID {row['catalog_id']}] {row['paragraph_topic']}\n"
        f"        {row['section_topic']}\n"
        f"        Example: {row['example']}"
    )


VARIANTS = {
    "A_section_only":       render_A_section_only,
    "B_paragraph_section":  render_B_paragraph_section,
    "C_example_only":       render_C_example_only,
    "D_section_example":    render_D_section_example,
    "E_all_fields":         render_E_all,
}

# Preview each variant's first catalogue entry
for variant_id, renderer in VARIANTS.items():
    sample = renderer(catalogue.iloc[0])
    print(f"=== {variant_id} ===")
    print(sample)
    print()

## Step 3: Build prompts and structured output schema

The system prompt is variant-specific (different catalogue rendering),
the user prompt template and the output schema are shared across all
variants. The schema asks for `catalog_id` only, no reasoning, since we
want a lean run.

In [ ]:
def build_system_prompt(variant_renderer: Callable, catalogue: pd.DataFrame) -> str:
    catalogue_text = "\n\n".join(
        variant_renderer(row) for _, row in catalogue.iterrows()
    )
    return f"""You are an expert in ERP and SaaS contract analysis. Your task is to classify a given contract section into one of 76 predefined catalogue topics.

Each catalogue entry is identified by an integer ID (1 to 76). You must select the single ID that best matches what the contract section addresses semantically.

Contract sections are written in German.

Catalogue:

{catalogue_text}

Output exactly one ID between 1 and 76."""


USER_PROMPT_TEMPLATE = """Contract section to classify:

{section_text}

Which catalogue ID best matches this section?"""


class CatalogIdOnly(BaseModel):
    """Force-choice output schema without reasoning."""
    catalog_id: int = Field(
        description="The catalog_id of the best matching catalogue entry, between 1 and 76",
        ge=1,
        le=76,
    )

## Step 4: Classification function

Same shape as in Notebook 03's main evaluation, but without reasoning
field, so output tokens stay minimal.

In [ ]:
def classify_section(
    section_text: str,
    system_prompt: str,
    model: str = LLM_MODEL,
    seed: int = SEED,
) -> dict:
    """Single LLM call. Returns predicted catalog_id and token usage."""
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": USER_PROMPT_TEMPLATE.format(section_text=section_text)},
        ],
        response_format=CatalogIdOnly,
        seed=seed,
    )
    result = completion.choices[0].message.parsed
    return {
        "catalog_id": result.catalog_id,
        "input_tokens": completion.usage.prompt_tokens,
        "output_tokens": completion.usage.completion_tokens,
        "cached_tokens": getattr(completion.usage.prompt_tokens_details, "cached_tokens", 0)
                         if hasattr(completion.usage, "prompt_tokens_details") else 0,
    }

## Step 5: Smoke test on variant B

Verify everything works end-to-end on a single call before launching
the full 5-variant sweep.

In [ ]:
test_section = sections.iloc[0]
test_text = test_section["section_content"]
test_gold = test_section["catalog_id"]

system_prompt_B = build_system_prompt(VARIANTS["B_paragraph_section"], catalogue)
print(f"System prompt B length: {len(system_prompt_B):,} characters\n")

print(f"Section: {test_text[:200]}...")
print(f"Gold: {test_gold}\n")

result = classify_section(test_text, system_prompt_B)
print(f"Predicted: {result['catalog_id']} ({'correct' if result['catalog_id'] == test_gold else 'wrong'})")
print(f"Tokens: input={result['input_tokens']}, output={result['output_tokens']}, cached={result['cached_tokens']}")

## Step 6: Run all five variants

Each variant runs over all 64 sections sequentially. With 1 second
delay between calls, each variant takes roughly 1.5 minutes, total
about 7 to 8 minutes for all five.

Token usage and cost are tracked per variant for the cost analysis in
Chapter 8 (though only variant B-style pricing is used in production).

In [ ]:
section_texts = sections["section_content"].tolist()
y_true = sections["catalog_id"].values

variant_results = {}
variant_costs = {}

for variant_id, renderer in VARIANTS.items():
    print(f"\n=== {variant_id} ===")
    system_prompt = build_system_prompt(renderer, catalogue)
    print(f"System prompt: {len(system_prompt):,} chars")
    
    predictions = []
    input_tok = output_tok = cached_tok = 0
    
    start = time.time()
    for i, text in enumerate(section_texts):
        r = classify_section(text, system_prompt)
        predictions.append(r["catalog_id"])
        input_tok += r["input_tokens"]
        output_tok += r["output_tokens"]
        cached_tok += r["cached_tokens"]
        if (i + 1) % 16 == 0 or (i + 1) == len(section_texts):
            print(f"  {i + 1}/{len(section_texts)} done")
        if i < len(section_texts) - 1:
            time.sleep(DELAY_BETWEEN_CALLS)
    elapsed = time.time() - start
    
    predictions = np.array(predictions)
    acc = accuracy_score(y_true, predictions)
    f1m = f1_score(y_true, predictions, average="macro", zero_division=0)
    f1w = f1_score(y_true, predictions, average="weighted", zero_division=0)
    
    # GPT-5.5 pricing: $5/M input, $30/M output, $2.5/M cached input
    cost = (
        (input_tok - cached_tok) * 5.0 / 1_000_000
        + cached_tok * 2.5 / 1_000_000
        + output_tok * 30.0 / 1_000_000
    )
    
    variant_results[variant_id] = {
        "predictions": predictions,
        "accuracy": acc,
        "f1_macro": f1m,
        "f1_weighted": f1w,
        "n_correct": int((predictions == y_true).sum()),
        "input_tokens": input_tok,
        "output_tokens": output_tok,
        "cached_tokens": cached_tok,
        "cost_usd": cost,
        "elapsed_seconds": elapsed,
    }
    variant_costs[variant_id] = cost
    
    print(f"  accuracy={acc:.4f}, f1_macro={f1m:.4f}, f1_weighted={f1w:.4f}")
    print(f"  tokens: in={input_tok:,} (cached={cached_tok:,}), out={output_tok:,}")
    print(f"  cost: ${cost:.4f}, elapsed: {elapsed:.1f}s")

total_cost = sum(variant_costs.values())
print(f"\n=== Total cost across all 5 variants: ${total_cost:.4f} ===")

## Step 7: Comparison table and selection

The selected variant is the one with the highest macro F1. In case of
ties or close calls, the a-priori-preferred variant B is chosen for
consistency with the theoretical argument.

In [ ]:
summary_rows = []
for variant_id, r in variant_results.items():
    summary_rows.append({
        "variant": variant_id,
        "n_correct": r["n_correct"],
        "accuracy": r["accuracy"],
        "f1_macro": r["f1_macro"],
        "f1_weighted": r["f1_weighted"],
        "input_tokens": r["input_tokens"],
        "output_tokens": r["output_tokens"],
        "cost_usd": r["cost_usd"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)
print("Comparison across prompt variants (sorted by f1_macro):\n")
print(summary_df.to_string(index=False))

selected_variant = summary_df.iloc[0]["variant"]
print(f"\nSelected variant (highest f1_macro): {selected_variant}")

## Step 8: Save outputs

In [ ]:
# Comparison summary
summary_df.to_csv(RESULTS_DIR / "variant_comparison.csv", index=False)

# Per-variant predictions
all_preds_df = pd.DataFrame({
    "contract": sections["contract"].values,
    "paragraph": sections["paragraph"].values,
    "section": sections["section"].values,
    "gold_catalog_id": sections["catalog_id"].values,
})
for variant_id, r in variant_results.items():
    all_preds_df[f"pred_{variant_id}"] = r["predictions"]
all_preds_df.to_csv(RESULTS_DIR / "all_variants_predictions.csv", index=False)

# Aggregated metadata
metadata = {
    "model": LLM_MODEL,
    "seed": SEED,
    "n_sections": int(len(sections)),
    "selected_variant": selected_variant,
    "variants": {
        v: {
            "accuracy": r["accuracy"],
            "f1_macro": r["f1_macro"],
            "f1_weighted": r["f1_weighted"],
            "n_correct": r["n_correct"],
            "cost_usd": r["cost_usd"],
        }
        for v, r in variant_results.items()
    },
    "total_cost_usd": total_cost,
}
with open(RESULTS_DIR / "prompt_selection_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved to results/llm_prompt_selection/:")
for f in sorted(RESULTS_DIR.iterdir()):
    if f.name != ".gitkeep":
        print(f"  {f.name}")

## Summary and handoff to Notebook 03

The selected variant from this run will be configured at the top of
Notebook 03 via the `SELECTED_VARIANT` constant. Open Notebook 03 and
set its `SELECTED_VARIANT` to the value printed in Step 7 above.

This Notebook 02 does not need to be re-run unless the catalogue or the
labelled sections change.